## Dataset

Download the fixed archive:

```bash
wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
sha256sum data.zip
unzip -q data.zip
```

The checksum must be:

```text
9e53453e3017502f22860cb08558f0cdb343e102fb61feab75960616185e7d67  data.zip
```

The archive contains a fixed training split and a fixed held-out evaluation
split. It has 1,002 image files. One GIF is kept in the archive for provenance
but `ImageFolder` ignores it, so the training run consumes 1,001 images:

| Split | Curly | Straight | Total |
| --- | ---: | ---: | ---: |
| Train | 410 | 390 | 800 |
| Held-out evaluation | 103 | 98 | 201 |

`torchvision.datasets.ImageFolder` assigns `curly=0` and `straight=1` because the class directories are sorted alphabetically. 

Do not create another random train/test split and do not augment the held-out evaluation images.

## Environment

For the reference run, use Python 3.11 and CPU wheels:

```bash
python -m pip install -r requirements-cpu.txt
```

The important CPU wheels are `torch==2.9.0+cpu` and `torchvision==0.24.0+cpu`. Also use `numpy==2.3.3`, and `Pillow==11.3.0`. 

The CPU reference defines the grading baseline. You can experiment on a GPU, but report the CPU run for the numeric questions.

The directory also contains `reference_train.py`. After extracting the archive, run `python reference_train.py --data-dir data` to produce both history files with the complete reference procedure.

## Reproducibility rules

Run this before constructing datasets, data loaders, the model, or the optimizer:

In [1]:
import random
import json

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cpu")

## Model and preprocessing

Use this transform for the baseline training set and, separately, for the held-out evaluation set:

In [2]:
evaluation_transform = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

In [3]:
train_dataset = datasets.ImageFolder("data/train", transform=evaluation_transform)
evaluation_dataset = datasets.ImageFolder("data/test", transform=evaluation_transform)

In [4]:
loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True,num_workers=0, generator=loader_generator)

evaluation_loader = DataLoader(evaluation_dataset, batch_size=20, shuffle=False, num_workers=0)

print(len(train_dataset), len(evaluation_dataset))
print(train_dataset.class_to_idx)

800 201
{'curly': 0, 'straight': 1}


In [5]:
class HairModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.hidden = nn.Linear(32 * 99 * 99, 64)
        self.output = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv(x)))
        x = x.flatten(start_dim=1)
        x = nn.functional.relu(self.hidden(x))
        return self.output(x)

In [6]:
model = HairModel().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8,
)

print("Parametri:", sum(p.numel() for p in model.parameters()))

Parametri: 20073473


## Question 1: loss

Which loss matches the model above?

- `nn.MSELoss()`
- `nn.BCEWithLogitsLoss()`
- `nn.CrossEntropyLoss()`
- `nn.CosineEmbeddingLoss()`

There is one intended answer: the model emits one logit and the loss applies the stable sigmoid-plus-binary-cross-entropy operation.

The answer is nn.BCEWithLogitsLoss()

## Question 2: number of parameters

What is the total number of trainable parameters? 

Count biases as well and use `sum(p.numel() for p in model.parameters())`.

The answer is 20073473

## Training metrics

Train for 10 epochs with the baseline transform. 

For every epoch, evaluate the held-out split with `model.eval()` and `torch.no_grad()`. 

Store these four series in a JSON file named `history_baseline.json`:
```python
{
    "train_loss": [...],
    "train_accuracy": [...],
    "evaluation_loss": [...],
    "evaluation_accuracy": [...],
}
```

For accuracy, use `(torch.sigmoid(logits) >= 0.5)`. Use the number of examples, not the number of batches, when averaging loss and accuracy.

In [7]:
def run_epoch(loader, train):
    model.train(train)                      # train=False equivale a model.eval()
    total_loss, correct, n = 0.0, 0, 0

    with torch.set_grad_enabled(train):     # train=False equivale a torch.no_grad()
        for x, y in loader:
            x = x.to(device)
            y = y.float().unsqueeze(1).to(device)   # shape (N, 1), float

            logits = model(x)
            loss = criterion(logits, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(x)      # media per esempi, non per batch
            preds = (torch.sigmoid(logits) >= 0.5).float()
            correct += (preds == y).sum().item()
            n += len(x)

    return total_loss / n, correct / n


def fit(epochs):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "evaluation_loss": [],
        "evaluation_accuracy": [],
    }
    for epoch in range(1, epochs + 1):
        tl, ta = run_epoch(train_loader, train=True)
        el, ea = run_epoch(evaluation_loader, train=False)
        history["train_loss"].append(tl)
        history["train_accuracy"].append(ta)
        history["evaluation_loss"].append(el)
        history["evaluation_accuracy"].append(ea)
        print(f"epoch {epoch:2d} | train {tl:.4f} {ta:.4f} | eval {el:.4f} {ea:.4f}")
    return history

In [9]:
history_baseline = fit(10)

with open("history_baseline.json", "w") as f:
    json.dump(history_baseline, f)

q3 = round(float(np.median(history_baseline["train_accuracy"])), 2)
q4 = round(float(np.std(history_baseline["train_loss"], ddof=0)), 3)
print("Q3:", q3)
print("Q4:", q4)

epoch  1 | train 0.5654 0.6900 | eval 0.6521 0.6567
epoch  2 | train 0.5185 0.7400 | eval 0.7122 0.6269
epoch  3 | train 0.4592 0.7825 | eval 0.6069 0.6766
epoch  4 | train 0.4113 0.8100 | eval 0.6733 0.6368
epoch  5 | train 0.3374 0.8538 | eval 0.6812 0.6915
epoch  6 | train 0.3378 0.8662 | eval 0.6986 0.6617
epoch  7 | train 0.3402 0.8225 | eval 0.6237 0.7363
epoch  8 | train 0.2059 0.9225 | eval 0.7282 0.6816
epoch  9 | train 0.1701 0.9263 | eval 0.9005 0.6965
epoch 10 | train 0.1570 0.9463 | eval 0.8294 0.7264
Q3: 0.84
Q4: 0.135


## Question 3: baseline training accuracy

Calculate the median of `train_accuracy` in `history_baseline.json`. 

Submit the value rounded to two decimal places. The grader accepts an absolute error of `0.05`; the answer is not selected from a fixed option list.

The answer is: 0.81


## Question 4: baseline training loss standard deviation

Calculate the population standard deviation of `train_loss` in `history_baseline.json` (for NumPy, use `np.std(values, ddof=0)`). 

Submit the value rounded to three decimal places. The grader accepts an absolute error of `0.02`.


The answer is: 0.139 su Chat, 0.135 su Claude


## Data augmentation

Keep the same model and optimizer. Replace only the training transform with:

In [10]:
train_transform_augmented = transforms.Compose([
    transforms.Resize((200, 200), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.RandomRotation(
        50,
        interpolation=transforms.InterpolationMode.NEAREST,
    ),
    transforms.RandomResizedCrop(
        200,
        scale=(0.9, 1.0),
        ratio=(0.9, 1.1),
        interpolation=transforms.InterpolationMode.BILINEAR,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_dataset = datasets.ImageFolder("data/train", transform=train_transform_augmented)
train_loader = DataLoader(
    train_dataset,
    batch_size=20,
    shuffle=True,
    num_workers=0,
    generator=loader_generator,      # lo stesso generator, senza reimpostarlo
)

history_augmented = fit(10)          # solo queste 10 epoche vanno nel JSON

with open("history_augmented.json", "w") as f:
    json.dump(history_augmented, f)

q5 = round(float(np.mean(history_augmented["evaluation_loss"])), 3)
q6 = round(float(np.mean(history_augmented["evaluation_accuracy"][-5:])), 2)
print("Q5:", q5)
print("Q6:", q6)

epoch  1 | train 0.6927 0.6050 | eval 0.6306 0.6866
epoch  2 | train 0.5892 0.6775 | eval 0.5701 0.7512
epoch  3 | train 0.5581 0.6600 | eval 0.5298 0.7214
epoch  4 | train 0.5418 0.7188 | eval 0.5309 0.7214
epoch  5 | train 0.5518 0.7025 | eval 0.5669 0.7164
epoch  6 | train 0.5108 0.7438 | eval 0.5631 0.7214
epoch  7 | train 0.4958 0.7412 | eval 0.5686 0.6866
epoch  8 | train 0.4788 0.7812 | eval 0.5775 0.7065
epoch  9 | train 0.4767 0.7700 | eval 0.5689 0.6866
epoch 10 | train 0.4806 0.7750 | eval 0.5510 0.7313
Q5: 0.566
Q6: 0.71


Do not recreate the model or optimizer. 

Train for 10 additional epochs and write only those ten epochs to `history_augmented.json`. 

Continue using the unchanged, deterministic `evaluation_dataset` and `evaluation_loader` for every evaluation.


## Question 5: augmented evaluation loss

Calculate the mean of `evaluation_loss` in `history_augmented.json`. 

Submit the value rounded to three decimal places; the grader accepts an absolute error of `0.02`.

The answer is: 0.575 per Chat, 0.611 per claude


## Question 6: augmented evaluation accuracy

Calculate the mean of the last five values in `evaluation_accuracy` in `history_augmented.json` (epochs 6 through 10 of the augmented phase). 

Submit the value rounded to two decimal places; the grader accepts an absolute error of `0.05`.

The tolerances cover minor CPU wheel differences, while the fixed seed, loader, split, architecture, and inference transform prevent the large uncontrolled variation from the 2025 exercise.

The answer is: 0.72

In [ ]:
import sys, torch, torchvision, numpy, PIL
print("python     ", sys.version.split()[0])
print("torch      ", torch.__version__)
print("torchvision", torchvision.__version__)
print("numpy      ", numpy.__version__)
print("Pillow     ", PIL.__version__)

In [ ]:
print("device usato:", device)
print("parametri del modello su:", next(model.parameters()).device)
print("CUDA disponibile:", torch.cuda.is_available())

In [ ]:
print("deterministic algorithms:", torch.are_deterministic_algorithms_enabled())
print("num_workers train/eval:", train_loader.num_workers, evaluation_loader.num_workers)
print("batch size:", train_loader.batch_size)
print("dataset:", len(train_dataset), len(evaluation_dataset))
print("classi:", train_dataset.class_to_idx)

In [ ]:
len(history_augmented["evaluation_loss"])

In [ ]:
print(len(history_baseline["train_loss"]), len(history_augmented["evaluation_loss"]))
print(np.mean(history_augmented["evaluation_loss"]))
print(history_augmented["evaluation_loss"])